# 03 - Correlation Study

**Purpose:** Identify which economic indicators lead food insecurity and estimate the lag time.

**Approach:** Since the Economic Fairness data (ending 2018) has limited overlap with Food Foundation data (starting 2020), we supplement with:
1. **ONS Claimant Count** — Monthly unemployment benefits claims (Nomis API)
2. **Energy Price Index** — CPI component for electricity/gas (ONS)

These contemporary shock signals can be correlated with food insecurity response.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import signal
from scipy.stats import pearsonr

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"

print(f"Project root: {PROJECT_ROOT}")

## 1. Fetch Claimant Count Data (Nomis API)

The Claimant Count measures the number of people claiming unemployment-related benefits.
This is a leading indicator — job losses precede food insecurity.

**Source:** [Nomis Web](https://www.nomisweb.co.uk/)

In [ ]:
def fetch_claimant_count_uk(start_year: int = 2019, end_year: int = 2025) -> pd.DataFrame:
    """
    Fetch UK-level Claimant Count data from Nomis API.
    
    The Claimant Count measures unemployment benefit claims and serves as
    a leading indicator for economic stress at household level.
    
    Args:
        start_year: First year to fetch
        end_year: Last year to fetch
    
    Returns:
        DataFrame with date and claimant_count columns
    """
    # Nomis API endpoint for Claimant Count (NM_162_1)
    # Geography 2092957697 = United Kingdom
    # Measures: 1 = count, 2 = rate
    
    base_url = "https://www.nomisweb.co.uk/api/v01/dataset/NM_162_1.json"
    
    params = {
        'geography': '2092957697',  # UK
        'sex': '7',  # Total (all)
        'age': '0',  # All ages 16+
        'measure': '1',  # Count
        'measures': '20100',  # Value
        'time': f'{start_year}01-{end_year}12',  # Monthly range
    }
    
    print(f"Fetching Claimant Count data from Nomis API...")
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        print(f"API error: {response.status_code}")
        print(f"URL: {response.url}")
        return None
    
    data = response.json()
    
    # Extract observations
    obs = data.get('obs', [])
    
    records = []
    for ob in obs:
        time_period = ob.get('time', {}).get('value', '')
        value = ob.get('obs_value', {}).get('value', None)
        
        if time_period and value:
            # Parse time period (format: "2020-01")
            try:
                date = datetime.strptime(time_period, '%Y-%m')
                records.append({'date': date, 'claimant_count': float(value)})
            except:
                pass
    
    if not records:
        print("No data returned from API")
        return None
    
    df = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
    print(f"Retrieved {len(df)} months of data")
    return df

In [ ]:
# Fetch claimant count data
claimant_df = fetch_claimant_count_uk(2019, 2025)

if claimant_df is not None:
    print(f"\nClaimant Count range: {claimant_df['date'].min()} to {claimant_df['date'].max()}")
    print(claimant_df.head())

### 1.1 Alternative: Manual Claimant Count Data

If the API doesn't return data, we can use known data points from ONS releases.

In [ ]:
# Fallback: Manual claimant count data from ONS publications
# Source: ONS Labour Market Statistics

MANUAL_CLAIMANT_DATA = {
    # Format: 'YYYY-MM': count (thousands)
    '2020-01': 1220,
    '2020-02': 1240,
    '2020-03': 1270,
    '2020-04': 2100,  # Pandemic spike
    '2020-05': 2800,
    '2020-06': 2700,
    '2020-07': 2700,
    '2020-08': 2700,
    '2020-09': 2600,
    '2020-10': 2600,
    '2020-11': 2600,
    '2020-12': 2600,
    '2021-01': 2600,
    '2021-04': 2500,
    '2021-07': 2100,
    '2021-10': 1800,
    '2022-01': 1600,
    '2022-04': 1500,
    '2022-07': 1450,
    '2022-09': 1450,  # Cost of living crisis period
    '2022-10': 1500,
    '2023-01': 1550,
    '2023-04': 1600,
    '2023-07': 1650,
    '2024-01': 1700,
    '2024-07': 1750,
    '2025-01': 1800,
}

if claimant_df is None or len(claimant_df) < 10:
    print("Using manual claimant count data...")
    records = [{'date': pd.to_datetime(k), 'claimant_count': v * 1000} 
               for k, v in MANUAL_CLAIMANT_DATA.items()]
    claimant_df = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
    print(f"Manual data: {len(claimant_df)} months")

## 2. Create Energy Price Index Proxy

Energy price shocks directly impact household budgets and precede food insecurity.
We'll create a proxy using known energy price events.

In [ ]:
# Energy price cap history (Ofgem announcements)
# Values are typical annual bills in GBP

ENERGY_PRICE_CAP = {
    '2020-01': 1179,
    '2020-04': 1126,  # Slight decrease
    '2020-10': 1042,
    '2021-04': 1138,
    '2021-10': 1277,  # Starting to rise
    '2022-04': 1971,  # 54% increase
    '2022-10': 2500,  # Government intervention (EPG)
    '2023-01': 2500,  # EPG extended
    '2023-04': 2500,
    '2023-07': 2074,  # Cap reduced
    '2023-10': 1834,
    '2024-01': 1928,
    '2024-04': 1690,
    '2024-07': 1568,
    '2024-10': 1717,
    '2025-01': 1738,
}

energy_records = [{'date': pd.to_datetime(k), 'energy_price_cap': v} 
                  for k, v in ENERGY_PRICE_CAP.items()]
energy_df = pd.DataFrame(energy_records).sort_values('date').reset_index(drop=True)

print(f"Energy price cap data: {len(energy_df)} periods")
print(f"Range: {energy_df['date'].min()} to {energy_df['date'].max()}")

## 3. Load Food Insecurity Data

In [ ]:
food_insecurity = pl.read_parquet(PROCESSED_DATA / "food_insecurity.parquet").to_pandas()
food_insecurity['date'] = pd.to_datetime(food_insecurity['survey_date'])

print(f"Food insecurity data: {len(food_insecurity)} surveys")
print(food_insecurity[['date', 'pct_food_insecure_all']].head())

## 4. Align Time Series

Resample all series to monthly frequency for correlation analysis.

In [ ]:
# Create common date range
date_range = pd.date_range('2020-01-01', '2025-01-31', freq='MS')

# Interpolate each series to monthly
def interpolate_to_monthly(df, date_col, value_col, target_dates):
    """Interpolate sparse time series to monthly frequency."""
    df = df.copy()
    df = df.set_index(date_col)
    df = df.reindex(df.index.union(target_dates))
    df[value_col] = df[value_col].interpolate(method='linear')
    return df.loc[target_dates][value_col].values

# Create aligned dataframe
aligned = pd.DataFrame({'date': date_range})

# Claimant count (already monthly-ish)
aligned['claimant_count'] = interpolate_to_monthly(
    claimant_df, 'date', 'claimant_count', date_range)

# Energy price cap (quarterly → monthly)
aligned['energy_price'] = interpolate_to_monthly(
    energy_df, 'date', 'energy_price_cap', date_range)

# Food insecurity (sparse survey dates)
aligned['food_insecurity'] = interpolate_to_monthly(
    food_insecurity, 'date', 'pct_food_insecure_all', date_range)

# Drop rows with NaN
aligned = aligned.dropna()

print(f"Aligned time series: {len(aligned)} months")
print(aligned.head(10))

## 5. Visualise Aligned Series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Claimant Count
axes[0].plot(aligned['date'], aligned['claimant_count'] / 1e6, 'o-', color='#2A9D8F', markersize=4)
axes[0].set_ylabel('Claimant Count\n(millions)')
axes[0].set_title('Economic Shock Signals vs Food Insecurity Response', fontsize=14)
axes[0].fill_between(aligned['date'], 0, aligned['claimant_count'] / 1e6, alpha=0.3, color='#2A9D8F')

# Energy Price
axes[1].plot(aligned['date'], aligned['energy_price'], 'o-', color='#E76F51', markersize=4)
axes[1].set_ylabel('Energy Price Cap\n(GBP/year)')
axes[1].fill_between(aligned['date'], aligned['energy_price'].min(), aligned['energy_price'], alpha=0.3, color='#E76F51')
axes[1].axhline(y=2500, color='red', linestyle='--', alpha=0.5, label='Crisis peak')

# Food Insecurity
axes[2].plot(aligned['date'], aligned['food_insecurity'], 'o-', color='#264653', markersize=5)
axes[2].set_ylabel('Food Insecurity\n(%)')
axes[2].set_xlabel('Date')
axes[2].fill_between(aligned['date'], 0, aligned['food_insecurity'], alpha=0.3, color='#264653')

axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig(OUTPUTS / 'aligned_series.png', dpi=150)
plt.show()

## 6. Cross-Correlation Analysis

Compute cross-correlation between shock signals and food insecurity at various lags.
Negative lag means the shock **leads** the response.

In [ ]:
def cross_correlation(x, y, max_lag=12):
    """
    Compute cross-correlation between two time series at various lags.
    
    Positive lag: x leads y (x happens first)
    Negative lag: y leads x
    
    For our analysis:
    - x = shock signal (claimant count, energy price)
    - y = response (food insecurity)
    - Positive lag means shock leads response (expected)
    
    Returns:
        lags: array of lag values in months
        correlations: correlation coefficient at each lag
    """
    # Normalize
    x = (x - x.mean()) / x.std()
    y = (y - y.mean()) / y.std()
    
    n = len(x)
    lags = np.arange(-max_lag, max_lag + 1)
    correlations = []
    
    for lag in lags:
        if lag > 0:
            # x leads y: compare x[:-lag] with y[lag:]
            corr, _ = pearsonr(x[:-lag], y[lag:])
        elif lag < 0:
            # y leads x: compare x[-lag:] with y[:lag]
            corr, _ = pearsonr(x[-lag:], y[:lag])
        else:
            corr, _ = pearsonr(x, y)
        correlations.append(corr)
    
    return lags, np.array(correlations)

In [ ]:
# Compute cross-correlations
max_lag = 6  # months

lags_claimant, corr_claimant = cross_correlation(
    aligned['claimant_count'].values, 
    aligned['food_insecurity'].values,
    max_lag=max_lag
)

lags_energy, corr_energy = cross_correlation(
    aligned['energy_price'].values, 
    aligned['food_insecurity'].values,
    max_lag=max_lag
)

print("Cross-correlation results:")
print(f"\nClaimant Count vs Food Insecurity:")
print(f"  Peak correlation: {corr_claimant.max():.3f} at lag {lags_claimant[corr_claimant.argmax()]} months")
print(f"\nEnergy Price vs Food Insecurity:")
print(f"  Peak correlation: {corr_energy.max():.3f} at lag {lags_energy[corr_energy.argmax()]} months")

In [ ]:
# Visualise cross-correlations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Claimant Count
axes[0].bar(lags_claimant, corr_claimant, color='#2A9D8F', alpha=0.7, edgecolor='black')
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Lag (months)')
axes[0].set_ylabel('Correlation')
axes[0].set_title('Claimant Count → Food Insecurity\n(positive lag = claimant leads)')
peak_lag = lags_claimant[corr_claimant.argmax()]
axes[0].annotate(f'Peak: {peak_lag} months', 
                 xy=(peak_lag, corr_claimant.max()),
                 xytext=(peak_lag + 1, corr_claimant.max() + 0.1),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=10, color='red')

# Energy Price
axes[1].bar(lags_energy, corr_energy, color='#E76F51', alpha=0.7, edgecolor='black')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Lag (months)')
axes[1].set_ylabel('Correlation')
axes[1].set_title('Energy Price → Food Insecurity\n(positive lag = energy leads)')
peak_lag = lags_energy[corr_energy.argmax()]
axes[1].annotate(f'Peak: {peak_lag} months', 
                 xy=(peak_lag, corr_energy.max()),
                 xytext=(peak_lag + 1, corr_energy.max() + 0.1),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 fontsize=10, color='red')

plt.tight_layout()
plt.savefig(OUTPUTS / 'cross_correlation.png', dpi=150)
plt.show()

## 7. Lag Estimation Summary

Extract the key findings for the policy report.

In [ ]:
print("="*60)
print("SYSTEMIC LATENCY ANALYSIS - KEY FINDINGS")
print("="*60)

# Claimant count analysis
peak_idx_claimant = corr_claimant.argmax()
peak_lag_claimant = lags_claimant[peak_idx_claimant]
peak_corr_claimant = corr_claimant[peak_idx_claimant]

print(f"\n1. CLAIMANT COUNT (Unemployment Shock)")
print(f"   Peak correlation: r = {peak_corr_claimant:.3f}")
print(f"   Optimal lag: {peak_lag_claimant} months")
if peak_lag_claimant > 0:
    print(f"   Interpretation: Unemployment spike LEADS food insecurity by ~{peak_lag_claimant} months")
elif peak_lag_claimant == 0:
    print(f"   Interpretation: Unemployment and food insecurity move together (no lead)")
else:
    print(f"   Interpretation: Food insecurity LEADS unemployment (unexpected)")

# Energy price analysis
peak_idx_energy = corr_energy.argmax()
peak_lag_energy = lags_energy[peak_idx_energy]
peak_corr_energy = corr_energy[peak_idx_energy]

print(f"\n2. ENERGY PRICE CAP (Cost-of-Living Shock)")
print(f"   Peak correlation: r = {peak_corr_energy:.3f}")
print(f"   Optimal lag: {peak_lag_energy} months")
if peak_lag_energy > 0:
    print(f"   Interpretation: Energy price rise LEADS food insecurity by ~{peak_lag_energy} months")
elif peak_lag_energy == 0:
    print(f"   Interpretation: Energy prices and food insecurity move together (no lead)")
else:
    print(f"   Interpretation: Food insecurity LEADS energy prices (unexpected)")

print(f"\n" + "="*60)
print("POLICY IMPLICATION")
print("="*60)

# Determine intervention window
intervention_window = max(peak_lag_claimant, peak_lag_energy, 0)
print(f"\nEstimated Intervention Window: {intervention_window} months")
print(f"\nIf economic stress indicators spike, policymakers have approximately")
print(f"{intervention_window} months to deploy preventive measures before")
print(f"food insecurity peaks in the affected population.")

## 8. Save Results

In [ ]:
# Save aligned data for further analysis
pl.from_pandas(aligned).write_parquet(PROCESSED_DATA / 'aligned_timeseries.parquet')

# Save correlation results
results = pd.DataFrame({
    'indicator': ['claimant_count', 'energy_price'],
    'peak_correlation': [peak_corr_claimant, peak_corr_energy],
    'optimal_lag_months': [peak_lag_claimant, peak_lag_energy],
})
results.to_csv(OUTPUTS / 'lag_estimates.csv', index=False)

print("Saved:")
print(f"  - {PROCESSED_DATA / 'aligned_timeseries.parquet'}")
print(f"  - {OUTPUTS / 'lag_estimates.csv'}")
print(f"\nNext: 04_lag_estimation.ipynb for Bayesian uncertainty quantification")